# 🛒 Sales Performance Analysis
### Python | SQL | Pandas | Matplotlib | Seaborn | Plotly

> **Author:** Hemalatha A N  
> **Dataset:** Global Superstore Sales Dataset  
> **Goal:** Uncover sales trends, revenue patterns and business insights

---

**Business Questions:**
- Which region generates the highest revenue?
- What are the top performing products and categories?
- Which months have highest sales?
- Which customer segment is most profitable?
- How does shipping mode affect profitability?

## 📦 Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('✅ Libraries imported successfully!')

## 📂 Step 2: Load & Explore Dataset

In [ ]:
# Load dataset
# Download from: https://www.kaggle.com/datasets/vivek468/superstore-dataset-final
df = pd.read_csv('superstore.csv', encoding='latin-1')

print('Dataset Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nData Types:')
print(df.dtypes)
df.head()

In [ ]:
# Basic statistics
print('📊 Dataset Summary:')
print(f'Total Orders: {df["Order ID"].nunique()}')
print(f'Total Customers: {df["Customer ID"].nunique()}')
print(f'Total Products: {df["Product ID"].nunique()}')
print(f'Date Range: {df["Order Date"].min()} to {df["Order Date"].max()}')
print(f'Total Revenue: ${df["Sales"].sum():,.2f}')
print(f'Total Profit: ${df["Profit"].sum():,.2f}')
print(f'\nMissing Values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

## 🧹 Step 3: Data Cleaning & Preprocessing

In [ ]:
# Convert dates
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

# Extract date features
df['Year']         = df['Order Date'].dt.year
df['Month']        = df['Order Date'].dt.month
df['Month Name']   = df['Order Date'].dt.strftime('%B')
df['Quarter']      = df['Order Date'].dt.quarter
df['Ship Days']    = (df['Ship Date'] - df['Order Date']).dt.days

# Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f'✅ Removed {before - len(df)} duplicates')
print(f'✅ Dataset ready: {df.shape}')

## 📊 Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# Sales by Region
region_sales = df.groupby('Region')[['Sales', 'Profit']].sum().sort_values('Sales', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

region_sales['Sales'].plot(kind='bar', ax=axes[0], color=['#0077b6','#00b4d8','#90e0ef','#caf0f8'])
axes[0].set_title('💰 Sales by Region', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Region')
axes[0].set_ylabel('Total Sales ($)')
axes[0].tick_params(axis='x', rotation=0)

region_sales['Profit'].plot(kind='bar', ax=axes[1], color=['#2d6a4f','#40916c','#52b788','#74c69d'])
axes[1].set_title('📈 Profit by Region', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Total Profit ($)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('sales_by_region.png', dpi=150, bbox_inches='tight')
plt.show()
print(region_sales)

In [ ]:
# Sales Trend Over Time
monthly_sales = df.groupby(['Year', 'Month'])['Sales'].sum().reset_index()
monthly_sales['Date'] = pd.to_datetime(monthly_sales[['Year', 'Month']].assign(Day=1))

plt.figure(figsize=(14, 5))
plt.plot(monthly_sales['Date'], monthly_sales['Sales'], color='#0077b6', linewidth=2.5, marker='o', markersize=4)
plt.fill_between(monthly_sales['Date'], monthly_sales['Sales'], alpha=0.1, color='#0077b6')
plt.title('📅 Monthly Sales Trend', fontweight='bold', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Total Sales ($)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('sales_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Category & Sub-Category Analysis
category_profit = df.groupby('Category')[['Sales', 'Profit']].sum()
category_profit['Profit Margin %'] = (category_profit['Profit'] / category_profit['Sales'] * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
axes[0].pie(category_profit['Sales'], labels=category_profit.index,
            autopct='%1.1f%%', colors=['#0077b6', '#00b4d8', '#90e0ef'], startangle=90)
axes[0].set_title('🏷️ Sales by Category', fontweight='bold', fontsize=14)

# Profit margin bar
category_profit['Profit Margin %'].plot(kind='bar', ax=axes[1], color=['#0077b6', '#00b4d8', '#90e0ef'])
axes[1].set_title('📊 Profit Margin by Category (%)', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Profit Margin (%)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('category_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(category_profit)

In [ ]:
# Top 10 Products by Sales
top_products = df.groupby('Product Name')['Sales'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
top_products.plot(kind='barh', color='#0077b6')
plt.title('🏆 Top 10 Products by Sales', fontweight='bold', fontsize=14)
plt.xlabel('Total Sales ($)')
plt.tight_layout()
plt.savefig('top_products.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Customer Segment Analysis
segment = df.groupby('Segment')[['Sales', 'Profit']].sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
segment['Sales'].plot(kind='pie', autopct='%1.1f%%', ax=axes[0],
                      colors=['#0077b6','#00b4d8','#90e0ef'])
axes[0].set_title('👥 Sales by Customer Segment', fontweight='bold')
axes[0].set_ylabel('')

segment['Profit'].plot(kind='bar', ax=axes[1], color=['#0077b6','#00b4d8','#90e0ef'])
axes[1].set_title('💰 Profit by Customer Segment', fontweight='bold')
axes[1].set_xlabel('Segment')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('segment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Shipping Mode Analysis
shipping = df.groupby('Ship Mode')[['Sales', 'Profit']].sum().sort_values('Sales', ascending=False)

plt.figure(figsize=(10, 5))
x = range(len(shipping))
width = 0.35
plt.bar([i - width/2 for i in x], shipping['Sales'], width, label='Sales', color='#0077b6')
plt.bar([i + width/2 for i in x], shipping['Profit'], width, label='Profit', color='#00b4d8')
plt.xticks(x, shipping.index)
plt.title('🚚 Sales & Profit by Shipping Mode', fontweight='bold', fontsize=14)
plt.legend()
plt.tight_layout()
plt.savefig('shipping_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 🗄️ Step 5: SQL Analysis

In [ ]:
# Load data into SQLite
conn = sqlite3.connect(':memory:')
df.to_sql('sales', conn, index=False, if_exists='replace')
print('✅ Data loaded into SQLite database!')

def run_query(query, title):
    result = pd.read_sql_query(query, conn)
    print(f'\n📊 {title}')
    print('-' * 50)
    print(result.to_string(index=False))
    return result

In [ ]:
# SQL Query 1: Revenue by Region
run_query("""
    SELECT Region,
           ROUND(SUM(Sales), 2) AS Total_Sales,
           ROUND(SUM(Profit), 2) AS Total_Profit,
           ROUND(SUM(Profit)/SUM(Sales)*100, 2) AS Profit_Margin
    FROM sales
    GROUP BY Region
    ORDER BY Total_Sales DESC
""", "Revenue & Profit by Region")

In [ ]:
# SQL Query 2: Top 5 Customers
run_query("""
    SELECT "Customer Name",
           COUNT(DISTINCT "Order ID") AS Total_Orders,
           ROUND(SUM(Sales), 2) AS Total_Sales,
           ROUND(SUM(Profit), 2) AS Total_Profit
    FROM sales
    GROUP BY "Customer Name"
    ORDER BY Total_Sales DESC
    LIMIT 5
""", "Top 5 Customers by Sales")

In [ ]:
# SQL Query 3: Monthly Sales Performance
run_query("""
    SELECT Year, Month,
           ROUND(SUM(Sales), 2) AS Monthly_Sales,
           ROUND(SUM(Profit), 2) AS Monthly_Profit,
           COUNT(DISTINCT "Order ID") AS Total_Orders
    FROM sales
    GROUP BY Year, Month
    ORDER BY Year, Month
    LIMIT 12
""", "Monthly Sales Performance")

In [ ]:
# SQL Query 4: Loss-making products
run_query("""
    SELECT "Product Name",
           ROUND(SUM(Sales), 2) AS Total_Sales,
           ROUND(SUM(Profit), 2) AS Total_Profit
    FROM sales
    GROUP BY "Product Name"
    HAVING Total_Profit < 0
    ORDER BY Total_Profit ASC
    LIMIT 10
""", "Top 10 Loss-Making Products")

conn.close()

## ✅ Step 6: Business Insights & Recommendations

In [ ]:
print('=' * 60)
print('📋 KEY BUSINESS INSIGHTS & RECOMMENDATIONS')
print('=' * 60)

print("""
1. 🏆 REGIONAL PERFORMANCE
   → West region leads in sales — increase marketing budget here
   → Central region has lowest profit margin — review pricing strategy

2. 📦 CATEGORY INSIGHTS
   → Technology has highest profit margin — prioritize tech products
   → Furniture has lowest margin — review supplier costs
   → Office Supplies has highest volume — good for steady revenue

3. 📅 SEASONAL TRENDS
   → Q4 (Oct-Dec) shows highest sales — stock up before holiday season
   → Q1 shows lowest sales — run promotions to boost revenue

4. 👥 CUSTOMER SEGMENTS
   → Consumer segment drives most sales — focus B2C marketing
   → Corporate segment most profitable per order — target B2B deals

5. 🚚 SHIPPING OPTIMIZATION
   → Standard Class most used but Second Class more profitable
   → Same Day shipping has lowest volume — promote for premium customers
""")

## 📊 Summary

| Metric | Value |
|--------|-------|
| Dataset | Global Superstore Sales |
| Analysis Type | EDA + SQL + Business Insights |
| Tools Used | Python, Pandas, Matplotlib, Seaborn, SQLite3 |
| Key Finding | West region & Technology category most profitable |

---
*Author: Hemalatha A N | Data Analyst Portfolio Project*